# DE5 Spark + Lakehouse 스타터 노트북

PySpark 3.5.6 (번들) 로컬 세션으로 Paimon / Iceberg 카탈로그를 탐색합니다.

- `paimon_lake.bronze.*` : Flink가 적재한 실시간/current-state 테이블 (`s3://paimon/warehouse`)
- `iceberg_lake.analytics.*` : Spark batch가 만든 분석 테이블 (Iceberg REST)

Iceberg/Paimon jar는 `spark-ivy-cache` 볼륨에서 재사용되므로 첫 세션 생성이 빠릅니다.

In [ ]:
from pyspark.sql import SparkSession

PAIMON_VERSION = "1.4.1"
ICEBERG_VERSION = "1.6.0"
PACKAGES = ",".join([
    f"org.apache.paimon:paimon-spark-3.5_2.12:{PAIMON_VERSION}",
    f"org.apache.paimon:paimon-s3:{PAIMON_VERSION}",
    f"org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:{ICEBERG_VERSION}",
    f"org.apache.iceberg:iceberg-aws-bundle:{ICEBERG_VERSION}",
])

spark = (
    SparkSession.builder.appName("de5-jupyter")
    .config("spark.jars.packages", PACKAGES)
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
            "org.apache.paimon.spark.extensions.PaimonSparkSessionExtensions")
    # --- Paimon (s3://paimon/warehouse) ---
    .config("spark.sql.catalog.paimon_lake", "org.apache.paimon.spark.SparkCatalog")
    .config("spark.sql.catalog.paimon_lake.warehouse", "s3://paimon/warehouse")
    .config("spark.sql.catalog.paimon_lake.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.paimon_lake.s3.access-key", "minioadmin")
    .config("spark.sql.catalog.paimon_lake.s3.secret-key", "minioadmin")
    .config("spark.sql.catalog.paimon_lake.s3.path.style.access", "true")
    # --- Iceberg (REST) ---
    .config("spark.sql.catalog.iceberg_lake", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.iceberg_lake.type", "rest")
    .config("spark.sql.catalog.iceberg_lake.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.iceberg_lake.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.iceberg_lake.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.iceberg_lake.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.iceberg_lake.s3.path-style-access", "true")
    .config("spark.sql.catalog.iceberg_lake.client.region", "us-east-1")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

## 1. Paimon bronze 테이블 (실시간/current-state)

In [ ]:
spark.sql("SHOW TABLES IN paimon_lake.bronze").show(truncate=False)
for t in ["ux_events_bronze", "review_current", "order_current"]:
    n = spark.sql(f"SELECT COUNT(*) c FROM paimon_lake.bronze.{t}").collect()[0]["c"]
    print(f"{t:20s} {n:>8,}")

In [ ]:
# upsert(current-state) 증거: order 마지막 상태 분포
spark.sql("""
  SELECT last_event_type, COUNT(*) AS cnt
  FROM paimon_lake.bronze.order_current
  GROUP BY last_event_type ORDER BY cnt DESC
""").show()

## 2. Paimon time-travel ($snapshots + scan.snapshot-id)

In [ ]:
# commit snapshot 목록
spark.sql("SELECT snapshot_id, commit_kind, total_record_count FROM paimon_lake.bronze.`review_current$snapshots` ORDER BY snapshot_id").show()
# 특정 snapshot 시점으로 읽기 (Spark는 VERSION AS OF 사용)
# spark.sql("SELECT COUNT(*) FROM paimon_lake.bronze.review_current VERSION AS OF 1").show()

## 3. Iceberg analytics 테이블 (batch BI)

In [ ]:
spark.sql("SHOW TABLES IN iceberg_lake.analytics").show(truncate=False)